# ChronoGRN demo — synthetic positive-control benchmark

This notebook loads one of the 16 **pretrained** checkpoints from
`results/benchmark/`, regenerates that seed's synthetic ground
truth and observed data, and reruns the *same* scoring function the
benchmark used (`score_model`) to confirm the live numbers match what's
recorded in the checkpoint and in `results/benchmark/results.json`
— i.e. this checkpoint is not just a snapshot, it's a working model that
reproduces its own reported metrics from scratch.

See `../README.md` for the full 16-seed summary and ablation table, and
`../docs/MODEL.md` for the model mechanism (delays, enhancer gating,
time-varying edges).


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from causal_grn.checkpoint import load_checkpoint
from bench.delayed_multiome import build_truth, simulate_dataset, split_train_heldout
from scripts.experiments.scoring import score_model

plt.rcParams["figure.facecolor"] = "white"
SEED = 0


## 1. Load the pretrained checkpoint

Self-contained: weights, constructor kwargs, TF-enhancer / enhancer-target
priors, and the recorded benchmark metrics for this seed.


In [ ]:
ckpt_path = ROOT / "results" / "benchmark" / f"seed{SEED}.pt"
model, payload = load_checkpoint(ckpt_path)
model.eval()

recorded_metrics = payload["metrics"]
n_params = sum(p.numel() for p in model.parameters())
print(f"Seed:      {SEED}")
print(f"TFs:       {model.n_tfs}   Enhancers: {model.n_enhancers}   Targets: {model.n_targets}")
print(f"Max delay (tau): {model.max_tau}   delay bins: {model.n_delay_bins}")
print(f"Dynamic edges: {model.dynamic_edges}   Enhancer gating: {model.enhancer_gating}")
print(f"Trainable parameters: {n_params:,}")
print()
print("Recorded metrics for this checkpoint:")
for k in ["edge_auprc", "dynamic_component_r", "access_delay_r", "target_delay_r",
          "heldout_perturbation_r", "heldout_target_rollout_r2"]:
    print(f"  {k}: {recorded_metrics[k]:.3f}")


## 2. Regenerate this seed's synthetic ground truth + data

`build_truth(seed=SEED)` deterministically constructs known edges, known
continuous delays, and known time-varying edge dynamics. `simulate_dataset`
then integrates *that* ground-truth model forward to produce noisy observed
trajectories — control, early/mid training knockdowns of TF0–TF3, and
held-out early/mid/late knockdowns of TF4/TF5 (`seed=SEED + 100`, matching
`run_optimized_benchmark.py::run_seed` exactly).


In [ ]:
truth = build_truth(seed=SEED)
data = simulate_dataset(truth, seed=SEED + 100)
train, heldout = split_train_heldout(data)

print(f"{len(data)} conditions: {len(train)} train, {len(heldout)} held-out")
print("Held-out (never used for fitting):", [data[i]['name'] for i in heldout])


## 3. Live scoring — reproduce the recorded metrics from scratch

This calls the exact `score_model` function the benchmark used. It
integrates the model forward (Heun steps) against the regenerated ground
truth and data, so a mismatch here would mean the checkpoint doesn't
actually reproduce its own published numbers.


In [ ]:
live_metrics = score_model(model, truth, data, heldout, use_reference=True, integration_step=0.008)

compare = ["edge_auprc", "dynamic_component_r", "access_delay_r", "target_delay_r",
           "heldout_perturbation_r", "heldout_target_rollout_r2"]
print(f"{'metric':<28}{'recorded':>10}{'live':>10}{'diff':>10}")
for k in compare:
    rec, live = recorded_metrics[k], live_metrics[k]
    print(f"{k:<28}{rec:>10.4f}{live:>10.4f}{abs(rec - live):>10.2e}")


## 4. Control trajectory reconstruction

Reference-conditioned rollout of the control condition (`data[0]`): the
vector field is corrected so the observed control trajectory is a solution
by construction, and held-out interventions are then scored as deviations
from it.


In [ ]:
times = torch.tensor(truth.times)
control_obs = data[0]["obs"]
model.set_reference_trajectory(times, torch.tensor(control_obs))
with torch.no_grad():
    control_pred = model.integrate(
        times, torch.tensor(data[0]["initial_state"]), max_step=0.008, method="heun"
    ).obs.numpy()
model.clear_reference()

fig, ax = plt.subplots(figsize=(6, 4))
target_slice = model.target_slice
var = np.var(control_obs[:, target_slice], axis=0)
top = np.argsort(-var)[:4]
colors = ["#4C3D9B", "#D45C2C", "#0F766E", "#1D6A9A"]
for i, gi in enumerate(top):
    ax.plot(truth.times, control_obs[:, target_slice][:, gi], color=colors[i], lw=2.0, label=f"target {gi}")
    ax.plot(truth.times, control_pred[:, target_slice][:, gi], color=colors[i], lw=1.2, ls="--")
ax.set_title(f"Control reconstruction (solid=observed, dashed=model)\nR2={live_metrics['control_target_r2']:.3f}")
ax.set_xlabel("Pseudotime"); ax.set_ylabel("Target state"); ax.legend(fontsize=8)
fig.tight_layout()


## 5. Delay recovery — learned vs. ground truth


In [ ]:
delays = model.get_delays()
access_pred = delays["access_to_tf"].detach().numpy()
access_true = truth.access_to_tf_delays

possible = truth.model.pathway_prior.detach().cpu().numpy() > 0
active_path = truth.active_edges & possible
target_pred = delays["tf_to_target"].detach().numpy()[active_path]
target_true = truth.tf_to_target_delays[active_path]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
lim = float(model.max_tau) * 1.05
axes[0].scatter(access_true, access_pred, s=40, c="#4C3D9B")
axes[0].plot([0, lim], [0, lim], "k--", lw=1)
axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
axes[0].set_xlabel("True access->TF delay"); axes[0].set_ylabel("Learned delay")
axes[0].set_title(f"Access delays  r={live_metrics['access_delay_r']:.2f}")

axes[1].scatter(target_true, target_pred, s=14, c="#D45C2C", alpha=0.6)
axes[1].plot([0, lim], [0, lim], "k--", lw=1)
axes[1].set_xlim(0, lim); axes[1].set_ylim(0, lim)
axes[1].set_xlabel("True TF->target delay"); axes[1].set_ylabel("Learned delay")
axes[1].set_title(f"Target delays  r={live_metrics['target_delay_r']:.2f}")
fig.tight_layout()


## 6. Held-out stage-specific knockdown

A TF4/TF5 knockdown timed at a pseudotime stage the model never trained on.
`intervention=` zeroes that TF's accessibility + RNA from the knockdown
start time; the model re-integrates conditioned on this condition's own
control-like reference to isolate the knockdown-driven deviation.


In [ ]:
example_idx = heldout[len(heldout) // 2]
example = data[example_idx]
print(f"Held-out condition: {example['name']}")

model.set_reference_trajectory(times, torch.tensor(control_obs))
with torch.no_grad():
    kd_pred = model.integrate(
        times, torch.tensor(example["initial_state"]), max_step=0.008,
        method="heun", intervention=example["intervention"],
    ).obs.numpy()
model.clear_reference()

true_delta = example["obs"][:, target_slice] - control_obs[:, target_slice]
pred_delta = kd_pred[:, target_slice] - control_pred[:, target_slice]
effect = np.argsort(-np.max(np.abs(true_delta), axis=0))[:3]

fig, ax = plt.subplots(figsize=(6, 4))
for i, gi in enumerate(effect):
    ax.plot(truth.times, true_delta[:, gi], color=colors[i], lw=2.0, label=f"target {gi} obs")
    ax.plot(truth.times, pred_delta[:, gi], color=colors[i], lw=1.2, ls="--", label=f"target {gi} pred")
ax.axhline(0, color="black", lw=0.6, ls=":")
ax.set_xlabel("Pseudotime"); ax.set_ylabel("Delta expression (KD - control)")
ax.set_title(f"{example['name']}  (held out from fitting)")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()


## 7. Paired ablations, across all 16 seeds

Loaded from `results/ablation/results.json` — same
architecture, one mechanism removed at a time, each refit and evaluated on
the same held-out design.


In [ ]:
abl = json.loads((ROOT / "results" / "ablation" / "results.json").read_text())
variants = abl["summary"]["variants"]
names = ["ChronoGRN", "No delay", "No gate", "Fixed W"]
vals = [variants[n]["heldout_perturbation_r"]["mean"] for n in names]
errs = [variants[n]["heldout_perturbation_r"]["sd"] for n in names]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(names, vals, yerr=errs, color=["#4C3D9B", "#D45C2C", "#0F766E", "#1D6A9A"], capsize=4)
ax.set_ylabel("Held-out endpoint response r (mean +/- sd, n=16)")
ax.set_title("Ablations")
fig.tight_layout()


## Next steps

- **Refit from scratch:** the "Reproduce from scratch" section in
  `../README.md` gives the exact commands for all 16 seeds plus ablations
  and figure rendering.
- **Model internals:** `../docs/MODEL.md` and
  `../src/causal_grn/enhancer_model.py` (`EnhancerDelayGRN`).
- **Ground-truth generator:** `../bench/delayed_multiome.py` — read this to
  see exactly what "known" means in a positive-control benchmark like this
  one (and why it's a different, easier claim than real-tissue validation).
- **Claim boundary:** `../docs/MODEL.md` section 7 is explicit that this is
  a synthetic identifiability result, not biological validation.
